In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("C:\\Users\\erons\\saas-churn-prediction\\data\\processed\\churn_features.csv")
train_df.head()

,Account length,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total eve minutes,Total eve calls,Total night minutes,Total night calls,Total intl minutes,Total intl calls,Customer service calls,Churn,Total usage minutes,Total calls,Service call ratio
0,128,0,1,25,265.1,110,197.4,99,244.7,91,10.0,3,1,False,717.2,303,0.003300
1,107,0,1,26,161.6,123,195.5,103,254.4,103,13.7,3,1,False,625.2,332,0.003012
2,137,0,0,0,243.4,114,121.2,110,162.6,104,12.2,5,0,False,539.4,333,0.000000
3,84,1,0,0,299.4,71,61.9,88,196.9,89,6.6,7,2,False,564.8,255,0.007843
4,75,1,0,0,166.7,113,148.3,122,186.9,121,10.1,3,3,False,512.0,359,0.008357


## Prepare Features and Target

## Applying Feature Engineering to Test Data

To ensure consistency between training and testing, the same feature engineering steps applied to the training dataset are manually applied to the test dataset.

This ensures that both datasets have identical feature representations, which is critical for accurate model evaluation.

The test dataset is processed independently to avoid data leakage and to simulate real-world deployment conditions.

## Load Raw Test Data

In [2]:
test_df = pd.read_csv("C:\\Users\\erons\\saas-churn-prediction\\data\\raw\\churn-bigml-20.csv")

In [3]:
#Drop same columns as in training data
test_df = test_df.drop(columns=[
    "Total day charge",
    "Total eve charge",
    "Total night charge",
    "Total intl charge"
])

In [4]:
#drop the "state" column
test_df = test_df.drop(columns=["State"])

In [5]:
test_df = test_df.drop(columns=["Area code"])

In [6]:
test_df["International plan"] = test_df["International plan"].map({"No":0, "Yes":1})
test_df["Voice mail plan"] = test_df["Voice mail plan"].map({"No":0, "Yes":1})

In [7]:
test_df["Total usage minutes"] = (
    test_df["Total day minutes"] +
    test_df["Total eve minutes"] +
    test_df["Total night minutes"] +
    test_df["Total intl minutes"]
)

In [8]:
test_df["Total calls"] = (
    test_df["Total day calls"] +
    test_df["Total eve calls"] +
    test_df["Total night calls"] +
    test_df["Total intl calls"]
)

In [9]:
test_df["Service call ratio"] = test_df["Customer service calls"] / test_df["Total calls"]

In [10]:
#save processed test data
test_df.to_csv("C:\\Users\\erons\\saas-churn-prediction\\data\\processed\\churn_test_features.csv", index=False)

In [11]:
#Load test data
test_df = pd.read_csv("C:\\Users\\erons\\saas-churn-prediction\\data\\processed\\churn_test_features.csv")
test_df.head()

,Account length,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total eve minutes,Total eve calls,Total night minutes,Total night calls,Total intl minutes,Total intl calls,Customer service calls,Churn,Total usage minutes,Total calls,Service call ratio
0,117,0,0,0,184.5,97,351.6,80,215.8,90,8.7,4,1,False,760.6,271,0.003690
1,65,0,0,0,129.1,137,228.5,83,208.8,111,12.7,6,4,True,579.1,337,0.011869
2,161,0,0,0,332.9,67,317.8,97,160.6,128,5.4,9,4,True,816.7,301,0.013289
3,111,0,0,0,110.4,103,137.3,102,189.6,105,7.7,6,2,False,445.0,316,0.006329
4,49,0,0,0,119.3,117,215.1,109,178.7,90,11.1,1,1,False,524.2,317,0.003155


## Define Train and Test

In [12]:
X_train = train_df.drop("Churn", axis=1)
y_train = train_df["Churn"]

X_test = test_df.drop("Churn", axis=1)
y_test = test_df["Churn"]

## Train the Base Model

In [14]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=3000, solver="lbfgs")
)

model.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression(max_iter=3000))])

In [15]:
model.named_steps["logisticregression"].n_iter_

array([21], dtype=int32)

## Make Predictions

In [16]:
y_pred = model.predict(X_test)

## Evaluate the Model

In [17]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.88      0.97      0.92       572
        True       0.47      0.19      0.27        95

    accuracy                           0.85       667
   macro avg       0.68      0.58      0.59       667
weighted avg       0.82      0.85      0.83       667



Although the model achieves 85% accuracy, it performs poorly on the minority class with a recall of 0.19. This indicates severe class imbalance, and the model fails to capture the positive class effectively. I would try to address this using class weighting, resampling techniques, and threshold tuning